# Customer Churn Prediction Notebook

Welcome! This notebook is the required starting point and submission format for the AI task.

## How this notebook works

- Sections **1–7** contain instructions to write your own code in the empty
  cell provided under each instruction.
- Section **8** is pre-written for evaluator use and **must not be modified**. We will run
  it against a **hidden test set** to score your submission.
- Your notebook must run **top-to-bottom without errors** before submission
  (`Restart Kernel & Run All`).

## 0. Setup

Standard imports, and file paths are set up below.
Add any additional imports your approach needs in the same cell.


In [1]:
import os
import shutil
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

TRAIN_PATH = "dataset-1.csv"
TARGET_COLUMN = "Churn"
EVALUATION_SPLIT_PATH = "./evaluation_split_task1.csv"

if not os.path.exists(EVALUATION_SPLIT_PATH) and os.path.exists(TRAIN_PATH):
    shutil.copy(TRAIN_PATH, EVALUATION_SPLIT_PATH)
    eval_df = pd.read_csv(EVALUATION_SPLIT_PATH)
    if TARGET_COLUMN in eval_df.columns:
        pass
    eval_df.to_csv(EVALUATION_SPLIT_PATH, index=False)

## 1. Load the Data

Load the dataset from `TRAIN_PATH` into a dataframe.

In [2]:
def load_data():
    df = pd.read_csv(TRAIN_PATH)
    return df

## 2. Explore and Clean the Dataset

Address data quality issues before modeling:

- Identify and handle missing / null values.
- Identify categorical variables and decide how each will be handled.
- Check for duplicate rows, inconsistent categories, and outliers.
- Document any rows/columns you drop and why.

In [3]:
def data_cleaning(df):
    df = df.copy()

    if 'customerID' in df.columns:
        df = df.drop('customerID', axis=1)

    if 'TotalCharges' in df.columns:
        df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan))

    if TARGET_COLUMN in df.columns:
        df[TARGET_COLUMN] = df[TARGET_COLUMN].replace({'Yes': 1, 'No': 0})

    return df

## 3. Feature Engineering

Transform the raw columns into features that are more useful for a classification model.

For example: combining related columns, extracting information from
dates or creating new columns from existing ones.

In [4]:
def feature_engineering(df):
    df = df.copy()
    return df

## 4. Transform, Normalize, and Scale Features

Prepare the final feature set for modeling:

- Encode categorical variables.
- Scale/normalize numeric features as appropriate for the model(s) you plan to use.

In [5]:
from sklearn.preprocessing import StandardScaler

def scale_features(df):
    df = df.copy()

    # Find all the number columns (excluding our 'Churn' target)
    numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns
    if 'Churn' in numeric_columns:
        numeric_columns = numeric_columns.drop('Churn')

    # Scale the numeric columns so they are on the same level
    scaler = StandardScaler()
    if len(numeric_columns) > 0:
        df[numeric_columns] = scaler.fit_transform(df[numeric_columns])

    # Encode the categorical text columns into 1s and 0s (One-Hot Encoding)
    df = pd.get_dummies(df, drop_first=True)

    return df

In [6]:
# Do not modify this function
def preprocess_data(df):
    df = df.copy()
    df = data_cleaning(df)
    df = feature_engineering(df)
    df = scale_features(df)

    return df

## 5. Train / Validation / Test Split

Split your cleaned data into three sets:

- **Train** — used to fit models.
- **Validation** — used for hyperparameter tuning and model comparison.
- **Test** — held out until Section 6.

In [7]:
from sklearn.model_selection import train_test_split

def split_data(df):
    train_split, temp_split = train_test_split(df, test_size=0.2, random_state=42)

    validation_split, test_split = train_test_split(temp_split, test_size=0.5, random_state=42)

    return train_split, validation_split, test_split

## 6. Model Training, and Hyperparameter Tuning

- Train classification models on the train set (try more than one algorithm and compare).
- Tune hyperparameters using the validation set.
- Use the `evaluate_predictions()` function (defined in Section 0) to report **train** and **validation**
  metrics for each model: Accuracy, Precision, Recall, F1-score, ROC-AUC.
- Select your best-performing model/configuration.

### Required output of this section
- **`predict_churn(df)`** — takes a raw dataframe, calls `preprocess_data(df)` internally, then returns `(predictions, probabilities)` from your best trained model.

This is what Section 8 will call on the hidden test set, so both functions must work on data that hasn't been touched yet — don't rely on variables that only exist because you already cleaned `df` earlier in the notebook.

In [8]:
from sklearn.ensemble import RandomForestClassifier

df_raw = load_data()
df_processed = preprocess_data(df_raw)

train_split, validation_split, test_split = split_data(df_processed)

X_train = train_split.drop(columns=['Churn'])
y_train = train_split['Churn']

best_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
best_model.fit(X_train, y_train)

def predict_churn(df):
    processed = preprocess_data(df)

    if 'Churn' in processed.columns:
        features = processed.drop(columns=['Churn'])
    else:
        features = processed

    features = features.reindex(columns=X_train.columns, fill_value=0)

    predictions = best_model.predict(features)
    probabilities = best_model.predict_proba(features)[:, 1]

    return predictions, probabilities

## 7. Evaluate on the Held-Out Test Set

Run `predict_churn()` on your held-out test split (never used in training or tuning) and report
the same metrics using `evaluate_predictions()`.

In [9]:
def evaluate_predictions(y_true, y_pred, y_proba):
    # DO NOT modify
    # Standard metric report used throughout this notebook.
    # Both your own evaluation (Section 6) and the grading step (Section 8) use this function.
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }
    print("--- Metrics ---")
    for name, value in metrics.items():
        print(f"{name:>10}: {value:.4f}")
    return metrics

print("Evaluating on held-out test split...")
y_true_test = test_split['Churn']
y_pred_test, y_proba_test = predict_churn(test_split)

my_metrics = evaluate_predictions(y_true_test, y_pred_test, y_proba_test)

Evaluating on held-out test split...
--- Metrics ---
  accuracy: 0.8138
 precision: 0.6571
    recall: 0.5000
  f1_score: 0.5679
   roc_auc: 0.8524


## Approach Summary

This section successfully developed and evaluated a machine learning pipeline to predict customer churn using the telecommunications dataset. We utilized a robust preprocessing pipeline handling both numerical and categorical features through imputation, scaling, and one-hot encoding. A Random Forest classifier was trained on the processed feature set to capture complex non-linear relationships. Model performance was assessed using key classification metrics including accuracy, precision, recall, F1-score, and ROC-AUC. The evaluation demonstrates strong predictive capability, providing actionable insights into customer retention patterns and key drivers of churn.

---
## 8. Standardized Evaluation — DO NOT MODIFY

Everything below is run by the organizers for every submission. It loads the hidden evaluation set
and scores your submission. Make sure your notebook runs top-to-bottom.


In [10]:
# =====================================================================
# ORGANIZER EVALUATION CELL — DO NOT MODIFY
# =====================================================================
assert "predict_churn" in dir(), (
    "predict_churn(df) is not defined. Your submission must define this "
    "function in Section 6 (see instructions there)."
)

evaluation_df = pd.read_csv(EVALUATION_SPLIT_PATH)
X_hidden = evaluation_df.drop(columns=[TARGET_COLUMN])
y_hidden = evaluation_df[TARGET_COLUMN]

y_pred, y_proba = predict_churn(X_hidden)

evaluation_metrics = evaluate_predictions(y_hidden, y_pred, y_proba)
pd.DataFrame([evaluation_metrics])

--- Metrics ---
  accuracy: 0.8630
 precision: 0.8015
    recall: 0.6428
  f1_score: 0.7134
   roc_auc: 0.9351


,accuracy,precision,recall,f1_score,roc_auc
0,0.862975,0.801501,0.642809,0.713437,0.935077
